# Vascular Network Experiment Notebook

Interactive testing of **Space Colonization** and **ODC (Optimized Directed Colonization)** backends.

No GUI required — everything runs from pure Python.

## 1. Setup & Imports

In [ ]:
import sys, os
from pathlib import Path

ROOT = str(Path(".").resolve().parent)
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

from test.space_colonization_runner import run_space_colonization, run_space_colonization_dual_tree
from test.odc_runner import run_odc
from test.notebook_utils import (
    plot_network_2d,
    plot_network_3d,
    print_stats,
    compare_networks,
    compare_stats_table,
    network_to_dataframe,
    save_network_json,
)

import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

## 2. Space Colonization
### 2a. Single-inlet (cylinder domain)

In [ ]:
# Tissue sampling configuration for SC (used when 'tissue_sampling' is provided)
# strategy: 'uniform' | 'depth_biased' | 'radial_biased' | 'boundary_shell' | 'gaussian' | 'mixture'
tissue_sampling = {
    "enabled": True,
    "strategy": "uniform",
    "n_points": 10000,  # backend may override to >= num_attractors
    "depth_reference": {"mode": "face", "face": "top"},
    "depth_distribution": "power",
    "depth_power": 2.0,
    # For gaussian experiments, set below:
    "gaussian_mean": [0.0, 0.0, 0.0],
    "gaussian_sigma": [0.003, 0.003, 0.003],
}


In [ ]:
sc_params = {
    "domain_type": "cylinder",
    "domain_radius": 0.005,
    "domain_height": 0.010,
    "domain_center": [0.0, 0.0, 0.0],

    "inlet_position": [0.0, 0.0, 0.005],
    "inlet_radius": 0.001,
    "vessel_type": "arterial",

    "num_attractors": 600,
    "attraction_distance": 0.015,
    "kill_distance": 0.003,
    "step_size": 0.005,
    "max_iterations": 500,
    "max_steps": 500,
    "branch_angle_deg": 35.0,
    "directional_bias": 0.85,
    "max_deviation_deg": 25.0,

    "encourage_bifurcation": True,
    "max_children_per_node": 2,
    "bifurcation_probability": 0.7,
    "min_attractions_for_bifurcation": 6,
    "bifurcation_angle_threshold_deg": 55.0,

    "min_radius": 0.0001,
    "taper_factor": 0.95,

    "progress": True,
    "kdtree_rebuild_tip_every": 1,
    "kdtree_rebuild_all_nodes_every": 10,
    "stall_steps_per_inlet": 10,
    "interleaving_strategy": "round_robin",

    "check_collisions": True,
    "collision_clearance": 0.0002,
    "collision_merge_distance": 0.0003,

    "seed": 42,
    "num_outlets": 50,
    "apply_murray": True,
    "murray_exponent": 3.0,
    "terminal_radius": 0.0003,
    "tissue_sampling": tissue_sampling
}

sc_net, sc_stats = run_space_colonization(sc_params)
print_stats(sc_stats, "Space Colonization -- single inlet")


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
for ax, proj in zip(axes, ["xy", "xz", "yz"]):
    plot_network_2d(sc_net, projection=proj, ax=ax, title=f"SC single-inlet ({proj})")
plt.tight_layout()
plt.show()

In [ ]:
# Mesh synthesis + voxel repair preview (single tree)
from generation.ops.mesh.synthesis import synthesize_mesh, MeshSynthesisPolicy
mesh_policy = {
    "add_node_spheres": False,
    "cap_ends": True,
    "segments_per_circle": 16,
    "radius_clamp_min": 1e-4,
    "radius_clamp_max": None,
    "voxel_repair_synthesis": True,
    "voxel_repair_pitch": 1e-4,
    "voxel_repair_auto_adjust": True,
    "voxel_repair_max_steps": 4,
    "voxel_repair_step_factor": 1.5,
}
try:
    mp = MeshSynthesisPolicy.from_dict(mesh_policy) if hasattr(MeshSynthesisPolicy, 'from_dict') else MeshSynthesisPolicy(**mesh_policy)
    mesh, report = synthesize_mesh(sc_net, policy=mp)
    print({k: report.metadata.get(k) for k in ['vertex_count','face_count','is_watertight','voxel_repair_applied']})
except NameError:
    print('sc_net not defined in this session; run the SC single-tree cell first.')


In [ ]:
# View / export the SC single-tree mesh
import plotly.graph_objects as go

try:
    verts = mesh.vertices
    faces = mesh.faces
    fig_mesh = go.Figure(data=[
        go.Mesh3d(
            x=verts[:, 0], y=verts[:, 1], z=verts[:, 2],
            i=faces[:, 0], j=faces[:, 1], k=faces[:, 2],
            opacity=0.6,
            color='crimson',
            flatshading=True,
        )
    ])
    fig_mesh.update_layout(
        title='SC Single-Tree Mesh',
        scene=dict(aspectmode='data'),
        width=800, height=600,
    )
    fig_mesh.show()

    # Export to STL
    stl_path = 'sc_single_tree.stl'
    mesh.export(stl_path)
    print(f'Mesh exported to {stl_path}  ({len(verts)} verts, {len(faces)} faces, watertight={mesh.is_watertight})')
except NameError:
    print('mesh not defined; run the mesh synthesis cell first.')


In [ ]:
fig3d = plot_network_3d(sc_net, title="SC Single-Inlet 3D")
fig3d.show()

### 2b. Multi-inlet (blended mode)

In [ ]:
sc_multi_params = {
    **sc_params,
    "inlets": [
        {"position": [0.003, 0.0, 0.005], "radius": 0.0008},
        {"position": [-0.003, 0.0, 0.005], "radius": 0.0008},
        {"position": [0.0, 0.003, 0.005], "radius": 0.0008},
    ],
    "multi_inlet_mode": "blended",
    "multi_inlet_blend_sigma": 0.0,
    "max_inlets": 10,
    "num_attractors": 2000,
    "seed": 123,
}

sc_multi_net, sc_multi_stats = run_space_colonization(sc_multi_params)
print_stats(sc_multi_stats, "SC -- blended multi-inlet")
plot_network_2d(sc_multi_net, title="SC blended multi-inlet (xy)")
plt.show()

In [ ]:
fig3d = plot_network_3d(sc_multi_net, title="SC blended multi-inlet 3D")
fig3d.show()

### 2c. Multi-inlet (partitioned_xy mode)

In [ ]:
sc_part_params = {
    **sc_multi_params,
    "multi_inlet_mode": "partitioned_xy",
    "partitioned_directional_bias": 1.0,
    "partitioned_max_deviation_deg": 30.0,
    "partitioned_cone_angle_deg": 30.0,
    "partitioned_cylinder_radius": 0.001,
    "seed": 123,
}

sc_part_net, sc_part_stats = run_space_colonization(sc_part_params)
print_stats(sc_part_stats, "SC -- partitioned_xy multi-inlet")
plot_network_2d(sc_part_net, title="SC partitioned_xy (xy)")
plt.show()

In [ ]:
fig3d = plot_network_3d(sc_part_net, title="SC partitioned_xy 3D")
fig3d.show()

### 2d. Multi-inlet (forest mode)

In [ ]:
sc_forest_params = {
    **sc_multi_params,
    "multi_inlet_mode": "forest",
    "seed": 123,
}

sc_forest_net, sc_forest_stats = run_space_colonization(sc_forest_params)
print_stats(sc_forest_stats, "SC -- forest multi-inlet")
plot_network_2d(sc_forest_net, title="SC forest mode (xy)")
plt.show()

In [ ]:
fig3d = plot_network_3d(sc_forest_net, title="SC forest multi-inlet 3D")
fig3d.show()

### 2e. Parameter sweep -- num_attractors

In [ ]:
sweep_results = {}
for n_att in [500, 1000, 2000, 4000]:
    p = {**sc_params, "num_attractors": n_att, "seed": 42}
    net, stats = run_space_colonization(p)
    sweep_results[f"attractors={n_att}"] = (net, stats)

compare_networks(sweep_results, projection="xy")

In [ ]:
compare_stats_table({k: v[1] for k, v in sweep_results.items()})

In [ ]:
for label, (net, stats) in sweep_results.items():
    fig3d = plot_network_3d(net, title=f"SC {label} 3D")
    fig3d.show()

### 2f. Dual tree (blended, no merge-on-collision)

In [ ]:
sc_dual_params = {
    **sc_params,
    "inlets": [
        {"position": [0.0, 0.0, 0.005], "radius": 0.001},
        {"position": [0.0, 0.0, -0.005], "radius": 0.001},
    ],
    "multi_inlet_mode": "blended",
    "multi_inlet_blend_sigma": 0.0,
    "collision_clearance": 0.00002,
    "check_collisions": True,
    "step_size": 0.0002,
    "attraction_distance": 0.018,
    "kill_distance": 0.0008,
    "max_iterations": 1500,
    "max_steps": 1500,
    "bifurcation_probability": 0.85,
    "max_children_per_node": 2,
    "num_attractors": 300,
    "max_inlets": 10,
    "seed": 42,
    "progress": True,
    "apply_murray": True,
    "murray_exponent": 3.0,
    "terminal_radius": 0.0003,
    "tissue_sampling": tissue_sampling
}

sc_dual_net, sc_dual_stats = run_space_colonization(sc_dual_params)
print_stats(sc_dual_stats, "SC Dual Tree (blended, no merge-on-collision)")
plot_network_2d(sc_dual_net, title="SC Dual Tree - blended (xy)")
plt.show()


In [ ]:
fig3d = plot_network_3d(sc_dual_net, title="SC Dual Tree - blended 3D")
fig3d.show()

## 3. Optimized Directed Colonization (ODC)

ODC extends space colonization with **hierarchical tissue targeting** and **Murray's law post-propagation**.
Different tissue point distributions produce unique branching structures.
Hierarchical ordering directs growth: high-priority regions are reached first.

### 3a. ODC with auto-generated hierarchical levels

Auto-generates 3 priority levels: deep interior -> mid-range -> filler.
The network grows toward the center first, then expands outward.

In [ ]:
odc_params = {
    "domain_type": "cylinder",
    "domain_radius": 0.005,
    "domain_height": 0.010,
    "inlet_position": [0.0, 0.0, 0.005],
    "inlet_radius": 0.001,
    "vessel_type": "arterial",
    "influence_radius": 0.015,
    "kill_radius": 0.003,
    "step_size": 0.005,
    "max_steps": 500,
    "bifurcation_probability": 0.7,
    "max_children_per_node": 2,
    "taper_factor": 0.95,
    "auto_n_levels": 3,
    "auto_points_per_level": 200,
    "apply_murray": True,
    "murray_exponent": 3.0,
    "terminal_radius": 0.0003,
    "seed": 42,
}

odc_net, odc_stats = run_odc(odc_params)
print_stats(odc_stats, "ODC -- auto hierarchical levels")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
for ax, proj in zip(axes, ["xy", "xz", "yz"]):
    plot_network_2d(odc_net, projection=proj, ax=ax, title=f"ODC auto ({proj})")
plt.tight_layout()
plt.show()

In [ ]:
fig3d = plot_network_3d(odc_net, title="ODC Auto-Hierarchical 3D")
fig3d.show()

### 3b. Seed sweep -- unique structures from different tissue distributions

Each seed creates a different random tissue distribution, producing
unique but biologically valid branching patterns.

In [ ]:
odc_seed_results = {}
for seed in [1, 42, 123, 999]:
    p = {**odc_params, "seed": seed}
    net, stats = run_odc(p)
    odc_seed_results[f"seed={seed}"] = (net, stats)

compare_networks(odc_seed_results, projection="xy")
compare_stats_table({k: v[1] for k, v in odc_seed_results.items()})

In [ ]:
for label, (net, stats) in odc_seed_results.items():
    fig3d = plot_network_3d(net, title=f"ODC {label} 3D")
    fig3d.show()

### 3c. Explicit tissue levels -- directing growth via ordering

Define custom tissue levels to control WHERE the network grows first.
Level 0 (highest priority) is reached before level 1, which is reached before level 2.

In [ ]:
rng = np.random.default_rng(42)

level_0_pts = np.column_stack([
    rng.normal(0, 0.001, 50),
    rng.normal(0, 0.001, 50),
    rng.uniform(-0.004, -0.002, 50),
])

level_1_pts = np.column_stack([
    rng.uniform(-0.003, 0.003, 100),
    rng.uniform(-0.003, 0.003, 100),
    rng.uniform(-0.003, 0.001, 100),
])

level_2_pts = np.column_stack([
    rng.uniform(-0.004, 0.004, 200),
    rng.uniform(-0.004, 0.004, 200),
    rng.uniform(-0.004, 0.004, 200),
])

odc_explicit_params = {
    **odc_params,
    "tissue_levels": [
        {"priority": 0, "points": level_0_pts.tolist(), "label": "deep core",
         "weight": 1.0, "coverage_threshold": 0.8},
        {"priority": 1, "points": level_1_pts.tolist(), "label": "mid-range",
         "weight": 0.6, "coverage_threshold": 0.7},
        {"priority": 2, "points": level_2_pts.tolist(), "label": "outer filler",
         "weight": 0.3, "coverage_threshold": 0.5},
    ],
    "augment_with_filler": False,
    "seed": 42,
}

odc_explicit_net, odc_explicit_stats = run_odc(odc_explicit_params)
print_stats(odc_explicit_stats, "ODC -- explicit 3-level")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
for ax, proj in zip(axes, ["xy", "xz", "yz"]):
    plot_network_2d(odc_explicit_net, projection=proj, ax=ax, title=f"ODC explicit ({proj})")
plt.tight_layout()
plt.show()

In [ ]:
fig3d = plot_network_3d(odc_explicit_net, title="ODC Explicit Levels 3D")
fig3d.show()

### 3d. Tissue distribution comparison -- clustered vs spread

Tightly clustered level-0 points produce focused branching; spread-out points produce wider trees.

In [ ]:
rng2 = np.random.default_rng(7)

clustered_l0 = np.column_stack([
    rng2.normal(0.002, 0.0005, 80),
    rng2.normal(-0.002, 0.0005, 80),
    rng2.normal(-0.003, 0.0005, 80),
])

spread_l0 = np.column_stack([
    rng2.uniform(-0.004, 0.004, 80),
    rng2.uniform(-0.004, 0.004, 80),
    rng2.uniform(-0.004, 0.004, 80),
])

odc_clustered_params = {
    **odc_params,
    "tissue_levels": [
        {"priority": 0, "points": clustered_l0.tolist(), "label": "cluster",
         "weight": 1.0, "coverage_threshold": 0.8},
    ],
    "augment_with_filler": True,
    "filler_n_points": 300,
    "seed": 7,
}

odc_spread_params = {
    **odc_params,
    "tissue_levels": [
        {"priority": 0, "points": spread_l0.tolist(), "label": "spread",
         "weight": 1.0, "coverage_threshold": 0.8},
    ],
    "augment_with_filler": True,
    "filler_n_points": 300,
    "seed": 7,
}

odc_clustered_net, odc_clustered_stats = run_odc(odc_clustered_params)
odc_spread_net, odc_spread_stats = run_odc(odc_spread_params)

compare_networks({
    "Clustered L0": (odc_clustered_net, odc_clustered_stats),
    "Spread L0": (odc_spread_net, odc_spread_stats),
}, projection="xy")

compare_stats_table({
    "Clustered": odc_clustered_stats,
    "Spread": odc_spread_stats,
})

In [ ]:
fig3d_c = plot_network_3d(odc_clustered_net, title="ODC Clustered 3D")
fig3d_c.show()

fig3d_s = plot_network_3d(odc_spread_net, title="ODC Spread 3D")
fig3d_s.show()

## 4. Head-to-Head: Space Colonization vs ODC

Compare both algorithms on the same cylinder domain.
SC uses attractor-driven growth; ODC adds hierarchical tissue ordering + Murray radii.

In [ ]:
compare_networks({
    "Space Colonization": (sc_net, sc_stats),
    "ODC auto": (odc_net, odc_stats),
    "ODC explicit": (odc_explicit_net, odc_explicit_stats),
}, projection="xz")

In [ ]:
compare_stats_table({
    "SC": sc_stats,
    "ODC auto": odc_stats,
    "ODC explicit": odc_explicit_stats,
})

In [ ]:
fig3d_sc = plot_network_3d(sc_net, title="SC 3D (head-to-head)")
fig3d_sc.show()

fig3d_odc = plot_network_3d(odc_net, title="ODC auto 3D (head-to-head)")
fig3d_odc.show()

fig3d_odc_e = plot_network_3d(odc_explicit_net, title="ODC explicit 3D (head-to-head)")
fig3d_odc_e.show()

### 4b. Branching characteristics

Compare segment length and radius distributions side by side.

In [ ]:
df_sc = network_to_dataframe(sc_net)
df_odc = network_to_dataframe(odc_net)

fig, axes = plt.subplots(2, 2, figsize=(16, 10))

axes[0, 0].hist(df_sc["length_m"] * 1000, bins=30, color="steelblue", edgecolor="white", alpha=0.8)
axes[0, 0].set_xlabel("Segment length (mm)")
axes[0, 0].set_title("SC -- segment lengths")

axes[0, 1].hist(df_odc["length_m"] * 1000, bins=30, color="darkorange", edgecolor="white", alpha=0.8)
axes[0, 1].set_xlabel("Segment length (mm)")
axes[0, 1].set_title("ODC -- segment lengths")

axes[1, 0].hist(df_sc["mean_radius_m"] * 1e6, bins=30, color="steelblue", edgecolor="white", alpha=0.8)
axes[1, 0].set_xlabel("Mean radius (um)")
axes[1, 0].set_title("SC -- vessel radii")

axes[1, 1].hist(df_odc["mean_radius_m"] * 1e6, bins=30, color="darkorange", edgecolor="white", alpha=0.8)
axes[1, 1].set_xlabel("Mean radius (um)")
axes[1, 1].set_title("ODC -- vessel radii (Murray-propagated)")

plt.tight_layout()
plt.show()

## 5. Segment-Level Analysis

In [ ]:
df = network_to_dataframe(sc_net)
df.head(10)

In [ ]:
df["length_mm"] = df["length_m"] * 1000
df["mean_radius_um"] = df["mean_radius_m"] * 1e6

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].hist(df["length_mm"], bins=30, color="steelblue", edgecolor="white")
axes[0].set_xlabel("Segment length (mm)")
axes[0].set_ylabel("Count")
axes[0].set_title("Segment length distribution")

axes[1].hist(df["mean_radius_um"], bins=30, color="indianred", edgecolor="white")
axes[1].set_xlabel("Mean radius (um)")
axes[1].set_ylabel("Count")
axes[1].set_title("Vessel radius distribution")

plt.tight_layout()
plt.show()

## 6. Export

In [ ]:
save_network_json(sc_net, "sc_network.json")
save_network_json(odc_net, "odc_network.json")
print("Networks saved to sc_network.json and odc_network.json")

## 7. Sandbox

Use cells below for your own experiments.

## ODC v2 Feature Tests
Exercises anti-starburst branching, flexible tissue distributions,
multi-tree coordination, and high-branching stress tests.

In [ ]:
import sys, json, time
from pathlib import Path
sys.path.insert(0, str(Path(".").resolve().parent))

from test.odc_runner import run_odc
from test.space_colonization_runner import run_space_colonization
from generation.tissue.distributions import TissueDistributionSpec
from generation.ops.multi_tree_odc import run_multi_tree_odc, TreeConfig
from generation.tissue.hierarchical import TissueLevel, HierarchicalTissueSpec
from generation.core.domain import CylinderDomain
from generation.core.types import Point3D
import numpy as np

def summarize(net, label=""):
    n = len(net.nodes)
    s = len(net.segments)
    t = sum(1 for nd in net.nodes.values() if nd.node_type == "terminal")
    j = sum(1 for nd in net.nodes.values() if nd.node_type == "junction")
    lengths = [seg.length for seg in net.segments.values()]
    mean_len = float(np.mean(lengths)) if lengths else 0.0
    print(f"{label}  nodes={n}  segments={s}  terminals={t}  junctions={j}  mean_seg={mean_len*1e6:.1f}um")


### Example 1 – Depth-biased tissue, tuned SC params
Increases attraction radius, lowers kill radius, bumps iterations.
Step size ~100 µm for fine branching that intensifies with depth.

In [ ]:
depth_params = {
    "domain_type": "cylinder",
    "domain_radius": 0.005,
    "domain_height": 0.010,
    "step_size": 0.0001,          # ~100 um steps
    "influence_radius": 0.020,    # larger attraction radius
    "kill_radius": 0.0005,        # smaller kill radius -> denser branching
    "max_steps": 2000,            # more iterations
    "bifurcation_probability": 0.90,
    "max_children_per_node": 3,
    "taper_factor": 0.92,
    "smoothing_weight": 0.4,
    "max_curvature_deg": 45.0,
    "min_clearance": 0.00002,
    "seed": 42,
}

# Generate depth-biased tissue points (power law, z-axis)
domain = CylinderDomain(radius=0.005, height=0.010, center=Point3D(0,0,0))
dist = TissueDistributionSpec(
    distribution_type="depth_biased",
    n_points=400,
    depth_axis=2,
    depth_power=2.5,            # concentrates points deeper
    depth_distribution="power",
    seed=42,
)
tissue_pts = dist.generate(domain)
depth_params["tissue_levels"] = [{
    "priority": 1,
    "points": tissue_pts.tolist(),
    "label": "depth_biased_power",
    "weight": 1.0,
    "coverage_threshold": 0.003,
}]

t0 = time.perf_counter()
net1, stats1 = run_odc(depth_params)
elapsed1 = time.perf_counter() - t0
summarize(net1, "Depth-biased")
print(f"  elapsed={elapsed1:.2f}s  iters={stats1['iterations_used']}")


### Example 1b – SC baseline with matched params (depth-biased comparison)
Same domain, step size, attraction/kill radius, bifurcation settings as the ODC
depth-biased example above. Uses uniformly scattered attractors instead of
depth-biased tissue levels, so differences reflect algorithm behaviour.

In [ ]:
sc_depth_params = {
    "domain_type": "cylinder",
    "domain_radius": 0.005,
    "domain_height": 0.010,
    "inlet_position": [0.0, 0.0, 0.005],
    "inlet_radius": 0.001,
    "vessel_type": "arterial",
    "step_size": 0.0001,
    "attraction_distance": 0.020,
    "kill_distance": 0.0005,
    "max_iterations": 2000,
    "max_steps": 2000,
    "encourage_bifurcation": True,
    "bifurcation_probability": 0.90,
    "max_children_per_node": 3,
    "taper_factor": 0.92,
    "num_attractors": 400,
    "seed": 42,
    "apply_murray": True,
    "murray_exponent": 3.0,
    "terminal_radius": 0.0003,
    "tissue_sampling": tissue_sampling
}

t0 = time.perf_counter()
sc_net1, sc_stats1 = run_space_colonization(sc_depth_params)
sc_elapsed1 = time.perf_counter() - t0
summarize(sc_net1, "SC depth-matched")
print(f"  elapsed={sc_elapsed1:.2f}s")


### Example 2 – Dual-tree (arterial + venous) with 3D Gaussian tissue
Two trees grow from opposite ends of the cylinder into a shared
Gaussian tissue cloud. Sequential strategy with collision avoidance.

In [ ]:
domain_mt = CylinderDomain(radius=0.005, height=0.010, center=Point3D(0, 0, 0))

# 3D Gaussian tissue cloud centred in the domain
gauss_dist = TissueDistributionSpec(
    distribution_type="gaussian",
    n_points=300,
    gaussian_centers=[(0.0, 0.0, 0.0)],
    gaussian_sigmas=[(0.003, 0.003, 0.003)],
    seed=7,
)
gauss_pts = gauss_dist.generate(domain_mt)
print(f"Gaussian tissue points: {gauss_pts.shape[0]}")

tissue_spec = HierarchicalTissueSpec(levels=[
    TissueLevel(priority=1, points=gauss_pts, label="gaussian",
               weight=1.0, coverage_threshold=0.003),
])

tree_cfgs = [
    TreeConfig(
        tree_id="arterial",
        vessel_type="arterial",
        inlet_position=[0.0, 0.0, 0.005],    # top
        inlet_radius=0.001,
        inlet_direction=[0, 0, -1],
        params={
            "step_size": 0.0002,
            "influence_radius": 0.018,
            "kill_radius": 0.0008,
            "max_steps": 1500,
            "bifurcation_probability": 0.85,
            "max_children_per_node": 2,
        },
    ),
    TreeConfig(
        tree_id="venous",
        vessel_type="venous",
        inlet_position=[0.0, 0.0, -0.005],   # bottom
        inlet_radius=0.001,
        inlet_direction=[0, 0, 1],
        params={
            "step_size": 0.0002,
            "influence_radius": 0.018,
            "kill_radius": 0.0008,
            "max_steps": 1500,
            "bifurcation_probability": 0.85,
            "max_children_per_node": 2,
        },
    ),
]

t0 = time.perf_counter()
mt_result = run_multi_tree_odc(
    domain=domain_mt,
    tissue_spec=tissue_spec,
    tree_configs=tree_cfgs,
    collision_radius=0.001,
    interleave_strategy="sequential",
    seed=42,
)
elapsed2 = time.perf_counter() - t0

for tid, net in mt_result.networks.items():
    summarize(net, f"  {tid}")
print(f"  collisions={mt_result.collision_count}  elapsed={elapsed2:.2f}s")


### Example 2b – SC dual-tree with matched params (Gaussian comparison)
Same domain, step size, attraction/kill radius, bifurcation settings as the ODC
dual-tree example above. Two inlets at top/bottom in blended mode.

In [ ]:
sc_dual_v2_params = {
    "domain_type": "cylinder",
    "domain_radius": 0.005,
    "domain_height": 0.010,
    "inlets": [
        {"position": [0.0, 0.0, 0.005], "radius": 0.001},
        {"position": [0.0, 0.0, -0.005], "radius": 0.001},
    ],
    "multi_inlet_mode": "blended",
    "multi_inlet_blend_sigma": 0.0,
    "collision_clearance": 0.00002,
    "check_collisions": True,
    "step_size": 0.0002,
    "attraction_distance": 0.018,
    "kill_distance": 0.0008,
    "max_iterations": 1500,
    "max_steps": 1500,
    "bifurcation_probability": 0.85,
    "max_children_per_node": 2,
    "num_attractors": 300,
    "max_inlets": 10,
    "seed": 42,
    "apply_murray": True,
    "murray_exponent": 3.0,
    "terminal_radius": 0.0003,
    "tissue_sampling": tissue_sampling
}

t0 = time.perf_counter()
sc_dual_v2_net, sc_dual_v2_stats = run_space_colonization(sc_dual_v2_params)
sc_dual_elapsed = time.perf_counter() - t0
summarize(sc_dual_v2_net, "SC dual-tree")
print(f"  elapsed={sc_dual_elapsed:.2f}s")


In [ ]:
# Mesh synthesis + voxel repair preview (dual tree)
from generation.ops.mesh.synthesis import synthesize_mesh, MeshSynthesisPolicy
mesh_policy_dual = {
    "add_node_spheres": False,
    "cap_ends": True,
    "segments_per_circle": 16,
    "radius_clamp_min": 1e-4,
    "radius_clamp_max": None,
    "voxel_repair_synthesis": True,
    "voxel_repair_pitch": 1e-4,
    "voxel_repair_auto_adjust": True,
    "voxel_repair_max_steps": 4,
    "voxel_repair_step_factor": 1.5,
}
try:
    mpd = MeshSynthesisPolicy.from_dict(mesh_policy_dual) if hasattr(MeshSynthesisPolicy, 'from_dict') else MeshSynthesisPolicy(**mesh_policy_dual)
    mesh2, report2 = synthesize_mesh(sc_dual_v2_net, policy=mpd)
    print({k: report2.metadata.get(k) for k in ['vertex_count','face_count','is_watertight','voxel_repair_applied']})
except NameError:
    print('sc_dual_v2_net not defined in this session; run the SC dual-tree v2 cell first.')


In [ ]:
# View / export the SC dual-tree mesh
import plotly.graph_objects as go

try:
    verts2 = mesh2.vertices
    faces2 = mesh2.faces
    fig_mesh2 = go.Figure(data=[
        go.Mesh3d(
            x=verts2[:, 0], y=verts2[:, 1], z=verts2[:, 2],
            i=faces2[:, 0], j=faces2[:, 1], k=faces2[:, 2],
            opacity=0.6,
            color='steelblue',
            flatshading=True,
        )
    ])
    fig_mesh2.update_layout(
        title='SC Dual-Tree Mesh',
        scene=dict(aspectmode='data'),
        width=800, height=600,
    )
    fig_mesh2.show()

    # Export to STL
    stl_path2 = 'sc_dual_tree.stl'
    mesh2.export(stl_path2)
    print(f'Mesh exported to {stl_path2}  ({len(verts2)} verts, {len(faces2)} faces, watertight={mesh2.is_watertight})')
except NameError:
    print('mesh2 not defined; run the dual-tree mesh synthesis cell first.')


### Example 3 – Branching stress test
~100 µm step, maximal branching, lots of iterations, depth-biased tissue.
Goal: push complexity to see how dense the tree can get.

In [ ]:
stress_params = {
    "domain_type": "cylinder",
    "domain_radius": 0.005,
    "domain_height": 0.010,
    "step_size": 0.0001,              # 100 um
    "influence_radius": 0.025,        # very large attraction
    "kill_radius": 0.00025,           # very small kill -> maximum branching
    "max_steps": 5000,                # lots of iterations
    "bifurcation_probability": 0.95,  # almost always branch
    "max_children_per_node": 4,       # up to quad-furcation
    "taper_factor": 0.90,
    "inlet_radius": 0.0015,
    "max_stall_steps": 60,
    "seed": 123,
}

# Depth-biased beta distribution -> more points deeper in tissue
stress_domain = CylinderDomain(radius=0.005, height=0.010, center=Point3D(0,0,0))
stress_dist = TissueDistributionSpec(
    distribution_type="depth_biased",
    n_points=800,
    depth_axis=2,
    depth_distribution="beta",
    depth_beta_params=(2.0, 5.0),   # skew toward deeper z values
    seed=123,
)
stress_pts = stress_dist.generate(stress_domain)
stress_params["tissue_levels"] = [{
    "priority": 1,
    "points": stress_pts.tolist(),
    "label": "depth_beta_stress",
    "weight": 1.0,
    "coverage_threshold": 0.002,
}]

print(f"Stress test: {stress_pts.shape[0]} tissue points")
t0 = time.perf_counter()
net_stress, stats_stress = run_odc(stress_params)
elapsed3 = time.perf_counter() - t0
summarize(net_stress, "Stress")
print(f"  elapsed={elapsed3:.2f}s  iters={stats_stress['iterations_used']}")
print(f"  levels_reached={stats_stress['levels_reached']}")


### Example 3b – SC branching stress test with matched params
Same domain, step size, attraction/kill radius, bifurcation settings as the ODC
stress test above. Tests how SC handles extreme branching parameters.

In [ ]:
sc_stress_params = {
    "domain_type": "cylinder",
    "domain_radius": 0.005,
    "domain_height": 0.010,
    "inlet_position": [0.0, 0.0, 0.005],
    "inlet_radius": 0.0015,
    "vessel_type": "arterial",
    "step_size": 0.0001,
    "attraction_distance": 0.025,
    "kill_distance": 0.00025,
    "max_iterations": 5000,
    "max_steps": 5000,
    "encourage_bifurcation": True,
    "bifurcation_probability": 0.95,
    "max_children_per_node": 4,
    "taper_factor": 0.90,
    "num_attractors": 800,
    "stall_steps_per_inlet": 60,
    "seed": 123,
    "apply_murray": True,
    "murray_exponent": 3.0,
    "terminal_radius": 0.0003,
    "tissue_sampling": tissue_sampling
}

print(f"SC stress test: {sc_stress_params['num_attractors']} attractors")
t0 = time.perf_counter()
sc_net_stress, sc_stats_stress = run_space_colonization(sc_stress_params)
sc_elapsed3 = time.perf_counter() - t0
summarize(sc_net_stress, "SC Stress")
print(f"  elapsed={sc_elapsed3:.2f}s")


### v2 Head-to-Head: SC vs ODC with matched parameters
Compare the SC and ODC runs above using the same configurations.

In [ ]:
print("=== Depth-biased config ===")
summarize(sc_net1, "  SC ")
summarize(net1, "  ODC")
print()
print("=== Dual-tree config ===")
summarize(sc_dual_v2_net, "  SC ")
for tid, net in mt_result.networks.items():
    summarize(net, f"  ODC-{tid}")
print()
print("=== Stress test config ===")
summarize(sc_net_stress, "  SC ")
summarize(net_stress, "  ODC")


In [ ]:
# Helper: compute minimum inter-tree clearance (m) for a dual-tree network
import numpy as np
def min_intertree_clearance(network, inlet_positions):
    inls = np.array(inlet_positions, dtype=float)  # shape (2,3) or (k,3)
    # assign each node to nearest inlet by Euclidean distance
    pts = []
    for n in network.nodes.values():
        pts.append([n.id, n.position.x, n.position.y, n.position.z])
    if not pts:
        return None
    arr = np.array(pts)
    coords = arr[:,1:4].astype(float)
    # compute distances node->each inlet
    dists = np.linalg.norm(coords[:,None,:] - inls[None,:,:], axis=2)
    groups = np.argmin(dists, axis=1)
    # pairwise distances between groups
    g0 = coords[groups==0]
    mins = []
    for gi in range(1, inls.shape[0]):
        gi_pts = coords[groups==gi]
        if len(g0)==0 or len(gi_pts)==0:
            continue
        # compute all pairwise distances efficiently
        # (|g0|,1,3) - (1,|gi|,3) -> (|g0|,|gi|,3)
        diff = g0[:,None,:] - gi_pts[None,:,:]
        d = np.linalg.norm(diff, axis=2)
        mins.append(float(np.min(d)))
    return None if not mins else min(mins)

# Example usage (uncomment and set inlet positions as used in your SC dual-tree cell):
# sc_min_clear = min_intertree_clearance(sc_dual_v2_net, [
#     [0.0, 0.0, -0.005],  # inlet A pos
#     [0.0, 0.0,  0.005],  # inlet B pos
# ])
# print('Min inter-tree clearance (µm):', 1e6 * sc_min_clear if sc_min_clear is not None else 'N/A')
